# Challenging SRL systems

Melina Paxinou \
Student ID: 2854344

Link to Logistic Regression model: https://drive.google.com/drive/folders/1NdbDuIgOkeQS959PQGJnTz6gD7whzhP_?usp=sharing \
Link to BERT model: https://drive.google.com/drive/folders/1bEp-e5BcYGgcubV4aEJr67UZnmDHJnh4?usp=sharing \
Please save the models in the same directory as this notebook.

## Imports

In [6]:
!pip install -r requirements.txt

In [7]:
import pandas as pd
import csv
import spacy
from spacy.tokens import Doc
from spacy import displacy
import benepar
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
import pickle
import networkx as nx
from benepar.spacy_plugin import BeneparComponent
from nltk.tree import Tree
import numpy as np
import re
from collections import Counter
from datasets import load_dataset
from evaluate import load
from datasets import Dataset
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
import random
import json
import utils_A1
from collections import defaultdict
from srl_bert_utils import compute_metrics

C:\Users\melou\anaconda3\Lib\site-packages\benepar\spacy_plugin.py:7: FutureWarning: BeneparComponent and NonConstituentException have been moved to the benepar module. Use `from benepar import BeneparComponent, NonConstituentException` instead of benepar.spacy_plugin. The benepar.spacy_plugin namespace is deprecated and will be removed in a future version.
  warnings.warn(
C:\Users\melou\anaconda3\Lib\site-packages\benepar\parse_chart.py:169: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loa

## Loading models

In [9]:
with open('logreg_model.pkl', 'rb') as f:
    logreg_model_pic = pickle.load(f)

# Load the vectorizer and assign it to a variable with '_pic'
with open('logreg_vec.pkl', 'rb') as f:
    logreg_vec_pic = pickle.load(f)

# Load the label encoder and assign it to a variable with '_pic'
with open('label_encoder.pkl', 'rb') as f:
    label_encoder_pic = pickle.load(f)

In [10]:
with open("srl_trainer.pkl", "rb") as model_file:
    pokemon_trainer = pickle.load(model_file)

In [11]:
# this assumes that the model and tokenizer folders are in the same directory as this notebook
pokemon_model = AutoModelForTokenClassification.from_pretrained("BERTModel")
pokemon_tokenizer = AutoTokenizer.from_pretrained("Tokenizer")

## Creating challenge set file

### Dative Alternation for ARG1

In [14]:
list_sentences_dative_pp = [
    ['She', 'gives', 'a', 'book', 'to', 'her', 'friend', '.'],
    ['He', 'lent', 'some', 'money', 'to', 'his', 'brother', '.'],
    ['They', 'promised', 'a', 'reward', 'to', 'her', '.'],
    ['He', 'offered', 'a', 'seat', 'to', 'the', 'guest', '.'],
    ['She', 'told', 'a', 'secret', 'to', 'her', 'friend', '.'],
    ['He', 'showed', 'the', 'photo', 'to', 'his', 'mother', '.'],
    ['They', 'sent', 'a', 'postcard', 'to', 'their', 'parents', '.'],
    ['She', 'mailed', 'the', 'documents', 'to', 'her', 'boss', '.'],
    ['He', 'threw', 'the', 'ball', 'to', 'his', 'friend', '.'],
    ['She', 'tossed', 'the', 'keys', 'to', 'him', '.'],
    ['He', 'brought', 'a', 'glass', 'of', 'water', 'to', 'her', '.'],
    ['They', 'emailed', 'the', 'report', 'to', 'her', '.'],
    ['He', 'faxed', 'the', 'contract', 'to', 'the', 'client', '.']
]


In [15]:
list_sentences_dative = [
    ['She', 'gives', 'her', 'friend', 'a', 'book', '.'],
    ['He', 'lent', 'his', 'brother', 'some', 'money', '.'],
    ['They', 'promised', 'her', 'a', 'reward', '.'],
    ['He', 'offered', 'the', 'guest', 'a', 'seat', '.'],
    ['She', 'told', 'her', 'friend', 'a', 'secret', '.'],
    ['He', 'showed', 'his', 'mother', 'the', 'photo', '.'],
    ['They', 'sent', 'their', 'parents', 'a', 'postcard', '.'],
    ['She', 'mailed', 'her', 'boss', 'the', 'documents', '.'],
    ['He', 'threw', 'his', 'friend', 'the', 'ball', '.'],
    ['She', 'tossed', 'him', 'the', 'keys', '.'],
    ['He', 'brought', 'her', 'a', 'glass', 'of', 'water', '.'],
    ['They', 'emailed', 'her', 'the', 'report', '.'],
    ['He', 'faxed', 'the', 'client', 'the', 'contract', '.']
]


In [16]:
list_sentences_pp = []
for sentence in list_sentences_dative_pp:
    joined_sentence = ' '.join(sentence)
    joined_sentence = re.sub(r'\s([?.!,¿])', r'\1', joined_sentence)
    list_sentences_pp.append(joined_sentence)

In [17]:
list_sentences_dat = []
for sentence in list_sentences_dative:
    joined_sentence = ' '.join(sentence)
    joined_sentence = re.sub(r'\s([?.!,¿])', r'\1', joined_sentence)
    list_sentences_dat.append(joined_sentence)

In [18]:
list_sentences_pp_pred = [
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0]
]

list_sentences_dative_pred = [
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0]
]


In [19]:
list_gold_arg1 = [['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], ['_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_'], ['_', '_', '_', '_', 'ARG1', '_', '_', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_', '_'], ['_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG1', '_'], ['_', '_', '_', 'ARG1', '_', '_', '_', '_']]

In [20]:
list_sentences_pp_arg1 = [
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_']
]

list_sentences_dative_arg1 = [
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_']
]

In [21]:
capability = 'Dative Alternation for ARG1'
list_dict_dat = []
checked_arg = 'ARG1 and PP'
test_type = 'MFT'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(list_sentences_dative_pp, list_sentences_pp, 
                                                    list_sentences_pp_pred, list_sentences_pp_arg1):
    cap_dict = {
        'capability': capability,
        'test_type': test_type,
        'sent_str': sent_str,
        'sent_tokens': sent_tok,
        'sent_predicate': sent_pred,
        'sent_gold': sent_gold,
        'sentence_number': sentence_number,  
        'checked_arg': checked_arg,
        'invariance': 'yes'
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

In [22]:
capability = 'Dative Alternation for ARG1'
checked_arg = 'ARG1 and Dative'
test_type = 'MFT'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(list_sentences_dative, list_sentences_dat, 
                                                    list_sentences_dative_pred, list_sentences_dative_arg1):
    cap_dict = {
        'capability': capability,
        'test_type': test_type,
        'sent_str': sent_str,
        'sent_tokens': sent_tok,
        'sent_predicate': sent_pred,
        'sent_gold': sent_gold,
        'sentence_number': sentence_number,  
        'checked_arg': checked_arg,
        'invariance': 'yes' 
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

### Dative Alternation for ARG2

In [24]:
list_gold_arg2 = [['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', 'ARG2', '_', '_', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', 'ARG2', '_'], ['_', '_', '_', 'ARG2', '_', '_', '_'], ['_', '_', '_', '_', '_', '_', 'ARG2', '_']]

In [25]:
list_sentences_pp_arg2 = [
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', 'ARG2', '_'],
    ['_', '_', '_', '_', '_', '_', 'ARG2', '_']
]

list_sentences_dative_arg2 = [
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', 'ARG2', '_', '_', '_', '_', '_'],
    ['_', '_', 'ARG2', '_', '_', '_'],
    ['_', '_', '_', 'ARG2', '_', '_', '_']
]


In [26]:
# for token, pred, gold in zip(list_sentences_dative, list_sentences_dative_pred, list_sentences_dative_arg2):
#     print(len(token), len(pred), len(gold))

In [27]:
capability = 'Dative Alternation for ARG2'
test_type = 'MFT'
checked_arg = 'ARG2-PP'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(list_sentences_dative_pp, list_sentences_pp, 
                                                    list_sentences_pp_pred, list_sentences_pp_arg2):
    cap_dict = {
        'capability': capability,
        'test_type': test_type,
        'sent_str': sent_str,
        'sent_tokens': sent_tok,
        'sent_predicate': sent_pred,
        'sent_gold': sent_gold,
        'sentence_number': sentence_number,  
        'checked_arg': checked_arg,
        'invariance': 'yes' 
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

In [28]:
capability = 'Dative Alternation for ARG2'
test_type = 'MFT'
checked_arg = 'ARG2-Dative'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(list_sentences_dative, list_sentences_dat, 
                                                    list_sentences_dative_pred, list_sentences_dative_arg2):
    cap_dict = {
        'capability': capability,
        'test_type': test_type,
        'sent_str': sent_str,
        'sent_tokens': sent_tok,
        'sent_predicate': sent_pred,
        'sent_gold': sent_gold,
        'sentence_number': sentence_number,  
        'checked_arg': checked_arg,
        'invariance': 'yes' 
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

### Causative-Inchoative Alternation for ARG1

In [30]:
caus_sent = [['The','sun','melts','the','butter','.'],
            ['I','broke','the','window','with','a','stone','.'],
            ['The','hacker','breaks','the','system','.'],
            ['The', 'chef', 'boiled', 'the', 'water', 'for', 'the', 'pasta', '.'],
            ['The', 'earthquake', 'shook', 'the', 'entire', 'building', '.'],
            ['He', 'loosened', 'the', 'cap', 'off', 'the', 'bottle', '.'],
            ['The', 'scientist', 'dissolved', 'the', 'substance', 'in', 'water', '.'],
            ['The', 'company', 'launched', 'the', 'new', 'product', 'last', 'week', '.'],
             ['The', 'artist', 'shattered', 'the', 'glass', 'sculpture', 'in', 'frustration', '.']]

In [31]:
inc_sent = [['The','butter','melts','.'],
            ['The','window','broke','.'],
            ['The','system','breaks','.'],
            ['The', 'water', 'boiled', 'rapidly', 'on', 'the', 'stove', '.'],
            ['The', 'building', 'shook', 'violently', 'during', 'the', 'earthquake', '.'],
            ['The', 'bottle', 'cap', 'loosened', 'unexpectedly', '.'],
            ['The', 'substance', 'dissolved', 'slowly', 'in', 'the', 'liquid', '.'],
            ['The', 'new', 'product', 'launched', 'successfully', '.'],
            ['The', 'glass', 'sculpture', 'shattered', 'into', 'tiny', 'pieces', '.']]

In [32]:
list_sentences_caus = []
for sentence in caus_sent:
    joined_sentence = ' '.join(sentence)
    joined_sentence = re.sub(r'\s([?.!,¿])', r'\1', joined_sentence)
    list_sentences_caus.append(joined_sentence)

In [33]:
list_sentences_inc = []
for sentence in inc_sent:
    joined_sentence = ' '.join(sentence)
    joined_sentence = re.sub(r'\s([?.!,¿])', r'\1', joined_sentence)
    list_sentences_inc.append(joined_sentence)

In [34]:
caus_gold = [['_', '_', '_', '_', 'ARG1', '_'], 
             ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], 
             ['_', '_', '_', '_', 'ARG1', '_'], 
             ['_', '_', '_', '_', 'ARG1', '_', '_', '_', '_'], 
             ['_', '_', '_', '_', '_', 'ARG1', '_'], 
             ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], 
             ['_', '_', '_', '_', 'ARG1', '_', '_', '_'], 
             ['_', '_', '_', '_', '_', 'ARG1', '_', '_', '_'], 
             ['_', '_', '_', '_', '_', 'ARG1', '_', '_', '_']]

inc_gold = [['_', 'ARG1', '_', '_'], 
            ['_', 'ARG1', '_', '_'], 
            ['_', 'ARG1', '_', '_'], 
            ['_', 'ARG1', '_', '_', '_', '_', '_', '_'], 
            ['_', 'ARG1', '_', '_', '_', '_', '_', '_'], 
            ['_', '_', 'ARG1', '_', '_', '_'], 
            ['_', 'ARG1', '_', '_', '_', '_', '_', '_'], 
            ['_', '_', 'ARG1', '_', '_', '_'], 
            ['_', '_', 'ARG1', '_', '_', '_', '_', '_']]


In [35]:
caus_pred = [[0, 0, 1, 0, 0, 0], 
             [0, 1, 0, 0, 0, 0, 0, 0], 
             [0, 0, 1, 0, 0, 0], 
             [0, 0, 1, 0, 0, 0, 0, 0, 0], 
             [0, 0, 1, 0, 0, 0, 0], 
             [0, 1, 0, 0, 0, 0, 0, 0], 
             [0, 0, 1, 0, 0, 0, 0, 0], 
             [0, 0, 1, 0, 0, 0, 0, 0, 0], 
             [0, 0, 1, 0, 0, 0, 0, 0, 0]]

inc_pred = [[0, 0, 1, 0], 
            [0, 0, 1, 0], 
            [0, 0, 1, 0], 
            [0, 0, 1, 0, 0, 0, 0, 0], 
            [0, 0, 1, 0, 0, 0, 0, 0], 
            [0, 0, 0, 1, 0, 0], 
            [0, 0, 1, 0, 0, 0, 0, 0], 
            [0, 0, 0, 1, 0, 0], 
            [0, 0, 0, 1, 0, 0, 0, 0]]


In [36]:
capability = 'Causative-Incohative Alternation ARG1'
checked_arg = 'Causative-ARG1'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(caus_sent,list_sentences_caus,caus_pred,caus_gold):
    cap_dict = {
    'capability':capability,
    'test_type': test_type,
    'checked_arg':checked_arg,
    'sent_str':sent_str,
    'sent_tokens': sent_tok,
    'sent_predicate': sent_pred,
    'sent_gold': sent_gold,
    'sentence_number': sentence_number,  
    'invariance': 'yes' 
}
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

In [37]:
capability = 'Causative-Incohative Alternation ARG1'
checked_arg = 'Inchoative-ARG1'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(inc_sent,list_sentences_inc,inc_pred,inc_gold):
    cap_dict = {
    'capability':capability,
    'test_type': test_type,
    'checked_arg':checked_arg,
    'sent_str':sent_str,
    'sent_tokens': sent_tok,
    'sent_predicate': sent_pred,
    'sent_gold': sent_gold,
    'sentence_number': sentence_number,  
    'invariance': 'yes' 
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

### Movement/Stative Verbs for ARG1

In [39]:
mov_arg1_sent = [
    ['Then', 'Paul', 'and', 'those', 'with', 'him', 'traveled', 'through', 'other', 'cities', '.'],
    
    ['The', 'shocks', 'generated', 'nerve', 'impulses', 'that', 'traveled', 'via', 'spine', 
     'to', 'brain', 'and', 'showed', 'up', 'clearly', 'on', 'a', 'brain-wave', 'monitor', ',', 
     'indicating', 'no', 'damage', 'to', 'the', 'delicate', 'spinal', 'tissue', '.'],
    
    ['His', 'foreign', 'travels', 'deeply', 'changed', 'his', 'perspective', 'on', 'the', 'world', '.'],
    
    ['Three', 'years', 'ago', ',', 'the', 'exchange', 'took', 'up', 'residence', 'in', 'a', 'space-age', 
     'tower', 'of', 'steel', 'and', 'glass', '--', 'evocative', 'of', 'the', 'kind', 'of', 
     'modern', 'architecture', 'that', 'Britain', "'s", 'Prince', 'Charles', 'has', 'denounced', '.'],
    
    ['During', 'the', 'time', 'of', 'his', 'residence', 'in', 'Mourhead', ',', 'Michael', 'Higgins', 
     'was', 'united', 'in', 'marriage', 'to', 'Agnes', 'Peterson', ','],
    
    ['Having', 'resided', 'in', 'the', 'great', 'state', 'of', 'California', 'for', 'the', 'past', 
     'seven', 'years', ',', 'I', 'find', 'it', 'hard', 'to', 'ignore', 'our', 'environmental', 
     'problems', '.'],
    
    ['Colleagues', 'today', 'recall', 'with', 'some', 'humor', 'how', 'meetings', 'would', 'crawl', 
     'into', 'the', 'early', 'morning', 'hours', '.'],
    
    ['John', 'dashed', 'home', 'through', 'the', 'park', '.']
]


In [40]:
mov_arg1_pred = [[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0], 
                 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
                 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], 
                 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
                 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
                 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
                 [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], 
                 [0, 1, 0, 0, 0, 0, 0]]

In [41]:
mov_arg1_gold = [['_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG1', '_'], 
                 ['_', '_', '_', '_', '_', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', 
                  '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
                 ['_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
                 ['_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG1', '_', '_', 
                  '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
                 ['_', '_', '_', '_', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
                 ['_', '_', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 
                  '_', '_', '_'], 
                 ['_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG1', '_'], 
                 ['_', '_', '_', '_', '_', 'ARG1', '_']]

In [42]:
gold = []
for sent in mov_arg1_sent:
    sent1 = []
    for idx,element in enumerate(sent):
        new_element = 0
        sent1.append(new_element)
    gold.append(sent1)

In [43]:
list_sentences_mov_arg1 = []
for sentence in mov_arg1_sent:
    joined_sentence = ' '.join(sentence)
    joined_sentence = re.sub(r'\s([?.!,¿])', r'\1', joined_sentence)
    list_sentences_mov_arg1.append(joined_sentence)

In [44]:
capability = 'Movement/Stative Verbs for ARG1'
checked_arg = 'ARG1'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(mov_arg1_sent,list_sentences_mov_arg1,mov_arg1_pred,mov_arg1_gold):
    cap_dict = {
    'capability':capability,
    'test_type': test_type,
    'checked_arg':checked_arg,
    'sent_str':sent_str,
    'sent_tokens': sent_tok,
    'sent_predicate': sent_pred,
    'sent_gold': sent_gold,
    'sentence_number': sentence_number,  
    'invariance': 'no' 
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

### Involuntary Experience/Change Intransitive Verbs for ARG1

In [46]:
die_verbs_sent = [
    ["They", "collapsed", "from", "exhaustion", "after", "running", "a", "marathon", "."],
    ["I", "exist", "in", "a", "world", "that", "constantly", "challenges", "me", "."],
    ["She", "remains", "calm", "even", "when", "everything", "around", "her", "is", "chaotic", "."],
    ["I", "shiver", "in", "the", "cold", "as", "the", "wind", "blows", "through", "my", "thin", "jacket", "."],
    ["They", "trembled", "with", "fear", "when", "they", "heard", "the", "strange", "noises", "outside", "."],
    ["She", "persists", "despite", "facing", "constant", "challenges", "in", "her", "career", "."],
    ["He", "quivers", "with", "nervousness", "before", "stepping", "onto", "the", "stage", "."],
    ["They", "deteriorate", "quickly", "without", "proper", "medical", "care", "and", "rest", "."],
    ["He", "died", "peacefully", "in", "his", "sleep", "after", "a", "long", "illness", "."],
    ["My", "son", "has", "grown", "taller", "over", "the", "past", "year", "."]
]

In [47]:
list_sentences_die = []
for sentence in die_verbs_sent:
    joined_sentence = ' '.join(sentence)
    joined_sentence = re.sub(r'\s([?.!,¿])', r'\1', joined_sentence)
    list_sentences_die.append(joined_sentence)

In [48]:
pred_die = [
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
]

In [49]:
gold_die = [['ARG1', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'], 
            ['_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_']]

In [50]:
capability = 'Involuntary Experience/Change Intransitive Verbs for ARG1'
checked_arg = 'ARG1'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(die_verbs_sent,list_sentences_die,pred_die,gold_die):
    cap_dict = {
    'capability':capability,
    'test_type': test_type,
    'checked_arg':checked_arg,
    'sent_str':sent_str,
    'sent_tokens': sent_tok,
    'sent_predicate': sent_pred,
    'sent_gold': sent_gold,
    'sentence_number': sentence_number,  
    'invariance': 'no' 
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

### Robustness: spelling errors on the predicate

In [52]:
list_sentences_robustness = [
    ['She', 'gies', 'a', 'book', 'to', 'her', 'friend', '.'],
    ['He', 'offerred', 'a', 'seat', 'to', 'the', 'guest', '.'],
    ['He', 'showwed', 'the', 'photo', 'to', 'his', 'mother', '.'],
    ['They', 'seent', 'a', 'postcard', 'to', 'their', 'parents', '.'],
    ['She', 'mailled', 'the', 'documents', 'to', 'her', 'boss', '.'],
    ['He', 'thraw', 'the', 'ball', 'to', 'his', 'friend', '.'],
    ['She', 'tosed', 'the', 'keys', 'to', 'him', '.'],
    ['He', 'broght', 'a', 'glass', 'of', 'water', 'to', 'her', '.'],
    ['They', 'emaileed', 'the', 'report', 'to', 'her', '.'],
    ['He', 'faaxed', 'the', 'contract', 'to', 'the', 'client', '.'],
    ['She', 'givees', 'her', 'friend', 'a', 'book', '.'],
    ['He', 'shoowed', 'his', 'mother', 'the', 'photo', '.'],
    ['They', 'sended', 'their', 'parents', 'a', 'postcard', '.'],
    ['She', 'maileds', 'her', 'boss', 'the', 'documents', '.'],
    ['She', 'tosse', 'him', 'the', 'keys', '.'],
    ['He', 'rought', 'her', 'a', 'glass', 'of', 'water', '.'],
    ['They', 'emailedd', 'her', 'the', 'report', '.'],
    ['He', 'faxeds', 'the', 'client', 'the', 'contract', '.'],
    ['The','sun','mels','the','butter','.'],
    ['I','brokes','the','window','with','a','stone','.'],
    ['The','hacker','breeaks','the','system','.'],
    ['The', 'company', 'launchd', 'the', 'new', 'product', 'last', 'week', '.'],
    ['The','window','brok','.'],
    ['The','system','bbreaks','.'],
    ['Then', 'Paul', 'and', 'those', 'with', 'him', 'travveled', 'through', 'other', 'cities', '.'],
    ["I", "exisst", "in", "a", "world", "that", "constantly", "challenges", "me", "."],
    ["She", "remainns", "calm", "even", "when", "everything", "around", "her", "is", "chaotic", "."],
    ["He", "ded", "peacefully", "in", "his", "sleep", "after", "a", "long", "illness", "."],
    ["My", "son", "has", "grewn", "taller", "over", "the", "past", "year", "."]    
]

In [53]:
list_sentences_rob = []
for sentence in list_sentences_robustness:
    joined_sentence = ' '.join(sentence)
    joined_sentence = re.sub(r'\s([?.!,¿])', r'\1', joined_sentence)
    list_sentences_rob.append(joined_sentence)

In [54]:
arg_rob = [
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', 'ARG1', '_', '_', '_', '_'],
    ['_', '_', '_', '_', 'ARG1', '_'],
    ['_', '_', '_', '_', '_', 'ARG1', '_', '_', '_'],
    ['_', 'ARG1', '_', '_'],
    ['_', 'ARG1', '_', '_'],
    ['_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARG1', '_'],
    ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_'],
    ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'],
    ['ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'],
    ['_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_']    
]


In [55]:
pred_rob = [
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
]


In [56]:
capability = 'Robustness: spelling errors in the predicate + ARG1'
checked_arg = 'ARG1'
test_type_inv = 'INV'
sentence_number = 1  

for sent_tok, sent_str, sent_pred, sent_gold in zip(list_sentences_robustness,list_sentences_rob,pred_rob,arg_rob):
    cap_dict = {
    'capability':capability,
    'test_type': test_type_inv,
    'checked_arg':checked_arg,
    'sent_str':sent_str,
    'sent_tokens': sent_tok,
    'sent_predicate': sent_pred,
    'sent_gold': sent_gold,
    'sentence_number': sentence_number,  
    'invariance': 'no' 
    }
    list_dict_dat.append(cap_dict)
    sentence_number += 1  

### Creating the json file

In [58]:
with open("./challenge_set.json", "w") as outfile:
     json.dump(list_dict_dat, outfile)

In [59]:
print(list_dict_dat[0])

{'capability': 'Dative Alternation for ARG1', 'test_type': 'MFT', 'sent_str': 'She gives a book to her friend.', 'sent_tokens': ['She', 'gives', 'a', 'book', 'to', 'her', 'friend', '.'], 'sent_predicate': [0, 1, 0, 0, 0, 0, 0, 0], 'sent_gold': ['_', '_', '_', 'ARG1', '_', '_', '_', '_'], 'sentence_number': 1, 'checked_arg': 'ARG1 and PP', 'invariance': 'yes'}


In [60]:
len(list_dict_dat)

117

## Standalone Logreg

In [62]:
def transform_to_test_sentence_logreg(sent, target, sentence_num):
    """
    Transforms a sentence and its corresponding target labels into a list of dictionaries.
    The dictionaries represent tokens and their respective features.
    First, it creates dictionaries with token, chosen predicate, and sentence number as keys, 
    which show whether the specific word is the predicate whose arguments need to be located.
    Then, it creates a dataframe of these dictionaries, and, lastly, passes it through the extract_features_from_dataframe 
    function and returns the feature list. 

    Parameters:
        sent (list): list of tokens in the sentence
        target (list): list of target labels indicating the chosen predicate (1 for predicate, 0 otherwise)

    Returns a list of dictionaries.
    """
    if len(sent) != len(target):
        raise ValueError(
            "The length of the sentence and the list to mark predicate must be the same.")

    # creates list of dicts for word and predicate and sentence number
    test_sentence = []
    all_predictions = []
    for token, label in zip(sent, target):
        test_sentence.append(
            {"token": token, "chosen_predicate": label, "sentence_num": sentence_num})

    # converts list of dicts to dataframe
    sentence_data = pd.DataFrame(test_sentence)

    # passes df to extract_features function 
    sentence_feats = utils_A1.extract_features_from_dataframe(sentence_data)

    # transforms the sentence features using the vectorizer
    sentence_feats_transformed = logreg_vec_pic.transform(sentence_feats)

    # makes predictions
    prediction_sent = logreg_model_pic.predict(sentence_feats_transformed)

    # inverse transforms the predicted labels
    original_labels = label_encoder_pic.inverse_transform(prediction_sent)
    
    all_predictions.append({
                "predicted_labels": original_labels,
                "sentence": sent,
                "raw_predictions": prediction_sent
            })
        
    return all_predictions

## Standalone BERT

In [64]:
def mega_function(sentence_data, pokemon_tokenizer):
    """
    Takes a dataframe that contains two columns: 'token' and 'chosen_predicate'. It executes all the preprocessing steps for BERT,
    i.e. creating the dictionary which contains the predicate values, the special tokens, the input_ids, attention_mask, 
    word_ids, tokenized_tokens, then it only keeps input_ids and attention_mask and creates the dataset that is used with BERT.

    Parameters:
    sentence_data (dataframe): dataframe created by a single sentence containing the columns 'token' and 'chosen_predicate'
    pokemon_tokenizer: custom tokenizer with added special predicate token

    Returns:
    tokenized_data (dictionary): contains all the necessary information to revert the sentence back into its original form
                                 after the subtokenization
    final_dataset (dataset): dataset created using the datasets library, readable by BERT
    """
    
    x = [
            {
            'tokens': sentence_data['token'].tolist(),
            'predicate': [0 if val == 0 else 1 for val in sentence_data['chosen_predicate']]
            }
        ]
    
    y = []
    for sentence in x:
        
        # extracting necessary lists
        tokens = sentence['tokens']
        predicates = sentence['predicate']
    
        # processing tokens based on the predicate
        processed_tokens = []
        processed_predicates = []
        for idx, (token, predicate) in enumerate(zip(tokens, predicates)):
            if predicate == 1:
                processed_predicates.append(predicates[idx])
                processed_tokens.append("[PRED]")

            processed_tokens.append(token)
            processed_predicates.append(predicates[idx])
    
        y.append({
            'tokens':processed_tokens,
            'predicates': processed_predicates
        })

    tokenized_data=[]
    
    for sent in y:
        single_sent={}
        tokenized_input = pokemon_tokenizer(sent["tokens"], is_split_into_words=True)
        tokens = pokemon_tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
        word_ids = tokenized_input.word_ids()
        
        single_sent["input_ids"] = tokenized_input["input_ids"]
        single_sent["attention_mask"] = tokenized_input["attention_mask"]
        single_sent['word_ids'] = word_ids
        single_sent['tokenized_tokens'] = tokens
        tokenized_data.append(single_sent)

    
    processed_data = []
    for example in tokenized_data:
        processed_data.append({
            "input_ids": tokenized_data[0]["input_ids"],
            "attention_mask": tokenized_data[0]["attention_mask"],
        })

        
    final_dataset = Dataset.from_dict({
        'input_ids': [example['input_ids'] for example in processed_data],
        'attention_mask': [example['attention_mask'] for example in processed_data]
    })

    return tokenized_data, final_dataset

In [65]:
def transform_to_test_sentence_bert(sent, target, pokemon_trainer, pokemon_model, pokemon_tokenizer):
    """
    Takes as input a list of strings (tokens) and a list of targets (strings of 0 and 1) and turns them into a dataframe
    with word and predicate columns (1 shows the predicate). Then, this dataframe is used in the mega_function, which executes 
    all the necessary preprocessing steps to create the final dataset suitable for BERT. 
    Finally, it predicts, then the special tokens are removed, the majority
    vote of the subtokens is kept, and the final list of predictions is created.

    Parameters:
    sent (list of strings): tokens that make up a sentence. 
    target (list of strings): list which contains as many 0 and 1 as the tokens of 'sent'. 1 represents the predicate.
    pokemon_trainer: trainer that uses the model and tokenizer to classify tokens
    pokemon_model: saved BERT model
    pokemon_tokenizer: custom tokenizer with added special predicate token

    Returns:
    A list of dictionaries.
    """
    if len(sent) != len(target):
        raise ValueError(
            "The length of the sentence and the list to mark cue must be the same.")

    label_to_id = {'ARG0': 45, 'ARG1': 0, 'ARG1-DSP': 1, 'ARG2': 17, 'ARG3': 20, 'ARG4': 16, 
                   'ARG5': 26, 'ARGA': 3, 'ARGM-ADJ': 9, 'ARGM-ADV': 19, 'ARGM-CAU': 18, 'ARGM-COM': 24, 
                   'ARGM-CXN': 5, 'ARGM-DIR': 21, 'ARGM-DIS': 15, 'ARGM-EXT': 46, 'ARGM-GOL': 8, 
                   'ARGM-LOC': 47, 'ARGM-LVB': 44, 'ARGM-MNR': 4, 'ARGM-MOD': 11, 'ARGM-NEG': 14, 
                   'ARGM-PRD': 10, 'ARGM-PRP': 23, 'ARGM-PRR': 13, 'ARGM-REC': 12, 'ARGM-TMP': 28, 
                   'C-ARG0': 22, 'C-ARG1': 51, 'C-ARG1-DSP': 58, 'C-ARG2': 30, 'C-ARG3': 25, 'C-ARG4': 56, 
                   'C-ARGM-ADV': 48, 'C-ARGM-COM': 35, 'C-ARGM-CXN': 2, 'C-ARGM-DIR': 29, 'C-ARGM-EXT': 52, 
                   'C-ARGM-GOL': 57, 'C-ARGM-LOC': 54, 'C-ARGM-MNR': 40, 'C-ARGM-PRP': 39, 'C-ARGM-PRR': 32, 
                   'C-ARGM-TMP': 31, 'C-V': 49, 'O': 27, 'R-ARG0': 43, 'R-ARG1': 55, 'R-ARG2': 33, 'R-ARG3': 37, 
                   'R-ARG4': 50, 'R-ARGM-ADV': 34, 'R-ARGM-CAU': 36, 'R-ARGM-COM': 53, 'R-ARGM-DIR': 6, 
                   'R-ARGM-GOL': 7, 'R-ARGM-LOC': 42, 'R-ARGM-MNR': 38, 'R-ARGM-TMP': 41}
    id_to_label = {v: k for k, v in label_to_id.items()}

    # creates list of dicts for token and chosen_predicate
    test_sentence = []
    for token, label in zip(sent, target):
        test_sentence.append(
            {"token": token, "chosen_predicate": token if label == 1 else 0})

    # converts list of dicts to dataframe
    sentence_data = pd.DataFrame(test_sentence)
    tokenized_data, final = mega_function(sentence_data, pokemon_tokenizer)

    # print(tokenized_data)
    predictions, _, _ = pokemon_trainer.predict(final)
    predictions = np.argmax(predictions, axis=2)
    
    for idx, i in enumerate(tokenized_data):
        length = len(i['word_ids'])
        i['predictions'] = predictions[idx][:length]

    processed_predictions = []
    all_predictions = []
    for data in tokenized_data:
        word_ids = data['word_ids']
        predictions = data['predictions']
        
        # filtering out elements where labels are -100
        word_ids = [b for a, b in zip(data['tokenized_tokens'], word_ids) if a not in ['[CLS]','[SEP]','[PAD]','[PRED]','[UNK]','[MASK]']]
        predictions = [b for a, b in zip(data['tokenized_tokens'], predictions) if a not in ['[CLS]','[SEP]','[PAD]','[PRED]','[UNK]','[MASK]']]

        # initializing variables to track the current group
        current_word_id = None
        current_predictions = []
        
        for idx, word_id in enumerate(word_ids):
            if word_id != current_word_id:
                # calculating majority vote for the previous group at the start of a new word id
                if current_predictions:
                    predicted_label_id = Counter(current_predictions).most_common(1)[0][0]
                    processed_predictions.append(id_to_label[predicted_label_id])  # index to label

                # reset for the new word_id
                current_word_id = word_id
                current_predictions = [predictions[idx]]
            else:
                # add to the current group
                current_predictions.append(predictions[idx])
        
        # handling the last group
        if current_predictions:
            predicted_label_id = Counter(current_predictions).most_common(1)[0][0]
            processed_predictions.append(id_to_label[predicted_label_id])  # index to label
    
        all_predictions.append({
            "sentence": sent,
            "raw_predictions": predictions,
            "processed_predictions": processed_predictions
        })
    
    return all_predictions

## Combined standalone

In [67]:
def compare_models(list_dict_dat, bert_model, bert_tokenizer, bert_trainer, logreg_model, logreg_vectorizer):
    """
    Processes each sentence with both BERT and Logistic Regression models,
    storing their predictions side by side in a DataFrame,
    keeping only relevant instances where gold_labels is not '_'.
    Also calculates the percentage of incorrect predictions (failure rate) for each model per label,
    and generates a table grouped by 'capability' and 'checked_arg'.
    """
    results = []
    stats = defaultdict(lambda: {'num_instances': 0, 'num_logreg_incorrect': 0, 'num_bert_incorrect': 0})
    
    for sentence in list_dict_dat:
        tokens = sentence['sent_tokens']
        predicate = sentence['sent_predicate']
        sentence_num = sentence['sentence_number']
        gold_labels = sentence['sent_gold']
        capability = sentence['capability']
        checked_arg = sentence['checked_arg']
        invariance = sentence['invariance']
        
        relevant_indices = [i for i, label in enumerate(gold_labels) if label != '_']
        
        bert_predictions = transform_to_test_sentence_bert(tokens, predicate, bert_trainer, bert_model, bert_tokenizer)
        logreg_predictions = transform_to_test_sentence_logreg(tokens, predicate, sentence_num)
        
        for bert_pred, logreg_pred in zip(bert_predictions, logreg_predictions):
            for idx, (token, bert_label, logreg_label, gold_label) in enumerate(zip(
                tokens, bert_pred['processed_predictions'], logreg_pred['predicted_labels'], gold_labels)):
                
                if idx not in relevant_indices:
                    continue
                
                results.append({
                    'sentence_number': sentence_num,
                    'token': token,
                    'bert_prediction': bert_label,
                    'logreg_prediction': logreg_label,
                    'gold_label': gold_label,
                    'capability': capability,
                    'checked_arg': checked_arg,
                    'invariance': invariance
                })
                
                key = (capability, checked_arg)
                stats[key]['num_instances'] += 1
                if bert_label != gold_label:
                    stats[key]['num_bert_incorrect'] += 1
                if logreg_label != gold_label:
                    stats[key]['num_logreg_incorrect'] += 1
    
    
    df_results = pd.DataFrame(results)
    df_results.to_csv("model_comparison_results.tsv", sep="\t", index=False)
    
    df_grouped = None
    bert_predictions_dict = {}
    logreg_predictions_dict = {}
    if 'yes' in df_results['invariance'].values:
        df_filtered = df_results[df_results['invariance'] == 'yes']
        df_grouped = df_filtered.groupby(['capability', 'checked_arg'], sort=False).agg(list)

        bert_predictions_dict = {
            (capability, checked_arg): row["bert_prediction"]
            for (capability, checked_arg), row in df_grouped.iterrows()
        }
        logreg_predictions_dict = {
            (capability, checked_arg): row["logreg_prediction"]
            for (capability, checked_arg), row in df_grouped.iterrows()
        }
        gold_labels_dict = {
            capability: [gold_label for idx, gold_label in enumerate(row["gold_label"])]
            for (capability, checked_arg), row in df_grouped.iterrows()
        }
        
    
    table_data = []
    for (capability, checked_arg), values in stats.items():
        num_instances = values['num_instances']
        num_logreg_incorrect = values['num_logreg_incorrect']
        num_bert_incorrect = values['num_bert_incorrect']
        
        logreg_failure_rate = round((num_logreg_incorrect / num_instances) * 100, 2) if num_instances > 0 else 0
        bert_failure_rate = round((num_bert_incorrect / num_instances) * 100, 2) if num_instances > 0 else 0
        
        table_data.append({
            'capability': capability,
            'checked_arg': checked_arg,
            'num_instances': num_instances,
            'num_logreg_incorrect': num_logreg_incorrect,
            'num_bert_incorrect': num_bert_incorrect,
            'logreg_failure_rate': logreg_failure_rate,
            'bert_failure_rate': bert_failure_rate
        })
    
    df_stats = pd.DataFrame(table_data)
    df_stats.to_csv("model_comparison_stats.tsv", sep="\t", index=False)

    return df_results, df_stats, bert_predictions_dict, logreg_predictions_dict, gold_labels_dict


In [68]:
df_inv, stats_inv, bert_predictions_dict, logreg_predictions_dict, gold_labels_dict = compare_models(list_dict_dat, pokemon_model, pokemon_tokenizer, pokemon_trainer, 
                                          logreg_model_pic, logreg_vec_pic)

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [69]:
df_inv

,sentence_number,token,bert_prediction,logreg_prediction,gold_label,capability,checked_arg,invariance
0,1,book,ARG1,ARG1,ARG1,Dative Alternation for ARG1,ARG1 and PP,yes
1,2,money,ARG1,O,ARG1,Dative Alternation for ARG1,ARG1 and PP,yes
2,3,reward,ARG1,ARG2,ARG1,Dative Alternation for ARG1,ARG1 and PP,yes
3,4,seat,ARG1,ARG1,ARG1,Dative Alternation for ARG1,ARG1 and PP,yes
4,5,secret,ARG2,ARG2,ARG1,Dative Alternation for ARG1,ARG1 and PP,yes
...,...,...,...,...,...,...,...,...
112,25,cities,ARGM-DIR,O,ARG1,Robustness: spelling errors in the predicate +...,ARG1,no
113,26,I,ARG0,O,ARG1,Robustness: spelling errors in the predicate +...,ARG1,no
114,27,She,ARG1,O,ARG1,Robustness: spelling errors in the predicate +...,ARG1,no
115,28,He,ARG1,O,ARG1,Robustness: spelling errors in the predicate +...,ARG1,no


In [70]:
stats_inv

,capability,checked_arg,num_instances,num_logreg_incorrect,num_bert_incorrect,logreg_failure_rate,bert_failure_rate
0,Dative Alternation for ARG1,ARG1 and PP,13,3,1,23.08,7.69
1,Dative Alternation for ARG1,ARG1 and Dative,13,4,0,30.77,0.00
2,Dative Alternation for ARG2,ARG2-PP,13,6,0,46.15,0.00
3,Dative Alternation for ARG2,ARG2-Dative,13,9,1,69.23,7.69
4,Causative-Incohative Alternation ARG1,Causative-ARG1,9,5,0,55.56,0.00
5,Causative-Incohative Alternation ARG1,Inchoative-ARG1,9,7,0,77.78,0.00
6,Movement/Stative Verbs for ARG1,ARG1,8,5,7,62.50,87.50
7,Involuntary Experience/Change Intransitive Ver...,ARG1,10,5,3,50.00,30.00
8,Robustness: spelling errors in the predicate +...,ARG1,29,27,2,93.10,6.90


In [71]:
logreg_predictions_dict

{('Dative Alternation for ARG1', 'ARG1 and PP'): ['ARG1',
  'O',
  'ARG2',
  'ARG1',
  'ARG2',
  'ARG1',
  'ARG1',
  'ARG1',
  'ARG1',
  'ARG1',
  'ARG1',
  'ARG1',
  'ARG1'],
 ('Dative Alternation for ARG1', 'ARG1 and Dative'): ['ARG1',
  'O',
  'ARG2',
  'ARG1',
  'ARG2',
  'ARG1',
  'ARG1',
  'ARG1',
  'O',
  'ARG1',
  'ARG1',
  'ARG1',
  'ARG1'],
 ('Dative Alternation for ARG2', 'ARG2-PP'): ['ARG2',
  'O',
  'ARG2',
  'O',
  'O',
  'O',
  'ARG2',
  'ARG2',
  'ARG2',
  'O',
  'ARG2',
  'ARG2',
  'O'],
 ('Dative Alternation for ARG2', 'ARG2-Dative'): ['ARG2',
  'O',
  'O',
  'O',
  'ARG2',
  'ARG2',
  'ARG2',
  'ARG1',
  'ARG1',
  'O',
  'O',
  'O',
  'ARG1'],
 ('Causative-Incohative Alternation ARG1', 'Causative-ARG1'): ['ARG1',
  'ARG1',
  'ARG1',
  'O',
  'O',
  'O',
  'O',
  'ARG1',
  'O'],
 ('Causative-Incohative Alternation ARG1', 'Inchoative-ARG1'): ['O',
  'ARG1',
  'ARG1',
  'O',
  'O',
  'O',
  'O',
  'ARG0',
  'O']}

In [72]:
def calculate_fail_rates(predictions_dict, gold_labels_dict):
    """
    Calculates failure rate per capability by comparing model predictions with gold labels for invariance tests.
    
    predictions_dict: Dictionary with (capability, checked_arg) as keys and lists of predictions as values.
    gold_labels_dict: Dictionary with capabilities as keys and lists of gold labels as values.
    
    Returns:
    Dictionary with failure rates per capability.
    """
    capability_failures = defaultdict(lambda: {"num_instances": 0, "num_failures": 0})

    grouped_predictions = defaultdict(dict)
    for (capability, checked_arg), pred_list in predictions_dict.items():
        grouped_predictions[capability][checked_arg] = pred_list

    for capability, checked_arg_dict in grouped_predictions.items():
        checked_args = list(checked_arg_dict.keys())

        if len(checked_args) <= 1:
            print(f"Skipping {capability}, expected at least 2 checked_args but found {len(checked_args)}")
            continue

        preds_list = [checked_arg_dict[arg] for arg in checked_args]
        gold = gold_labels_dict.get(capability, [])

        if any(len(preds) != len(gold) for preds in preds_list):
            print(f"Skipping {capability}, mismatched prediction/gold label lengths")
            continue

        num_instances = len(gold)
        num_failures = sum(
            any(p != gold_label for p in preds)  
            for *preds, gold_label in zip(*preds_list, gold)  
        )

        capability_failures[capability]["num_instances"] = num_instances
        capability_failures[capability]["num_failures"] = num_failures

    fail_rate_dict = {
        capability: round((values["num_failures"] / values["num_instances"]) * 100, 2)
        if values["num_instances"] > 0 else 0
        for capability, values in capability_failures.items()
    }

    return fail_rate_dict


In [73]:
fail_rates_logreg = calculate_fail_rates(logreg_predictions_dict, gold_labels_dict)
fail_rates_bert = calculate_fail_rates(bert_predictions_dict, gold_labels_dict)
print('Logistic Regression')
fail_rates_logreg

Logistic Regression


{'Dative Alternation for ARG1': 30.77,
 'Dative Alternation for ARG2': 84.62,
 'Causative-Incohative Alternation ARG1': 77.78}

In [74]:
print('BERT')
fail_rates_bert

BERT


{'Dative Alternation for ARG1': 7.69,
 'Dative Alternation for ARG2': 7.69,
 'Causative-Incohative Alternation ARG1': 0.0}

In [75]:
with open("fail_rates_inv.json", "w") as f:
    json.dump({"Logistic Regression": fail_rates_logreg, "BERT": fail_rates_bert}, f, indent=4)